# Can we predict hospital readmission?
Dataset: `data.csv` - 101,766 diabetic-patient hospital encounters. Target: `readmitted` (`NO`, `>30` days, `<30` days).

Two questions, because they are practically different:
1. **Any readmission** (`<30` or `>30` vs `NO`)
2. **Early readmission** (`<30` vs everything else) - the one hospitals are usually penalised for; much rarer.

Methodology choices that matter:
- The same patient appears in several encounters (`patient_nbr`), so all CV folds are **grouped by patient**. Otherwise the model is rewarded for recognising patients and scores are inflated.
- Patients who died or went to hospice cannot be readmitted; those encounters are removed.
- Metrics: ROC-AUC and PR-AUC (classes are imbalanced), compared against a trivial baseline.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, cross_val_predict, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
RS = 0
df = pd.read_csv('data.csv', na_values=['?'], low_memory=False)
print(df.shape)
df.readmitted.value_counts()

## 1. Data checks

In [ ]:
print('unique patients:', df.patient_nbr.nunique(), '| unique encounters:', df.encounter_id.nunique())
print((df.isna().mean()[lambda s: s > 0] * 100).round(1).to_string())
df.readmitted.value_counts(normalize=True).round(3)

`weight` (97% missing) is dropped. For `max_glu_serum` and `A1Cresult`, *missing = not measured* is itself informative, so it is kept as a category.

## 2. Cleaning & features

In [ ]:
# Remove encounters that end in death / hospice (cannot be readmitted)
dead_hospice = [11, 13, 14, 19, 20, 21]
d = df[~df.discharge_disposition_id.isin(dead_hospice)].copy()
print('removed', len(df) - len(d), 'death/hospice encounters')
d = d[d.gender != 'Unknown/Invalid']

y_any = (d.readmitted != 'NO').astype(int)
y_30 = (d.readmitted == '<30').astype(int)
groups = d.patient_nbr.values
print('P(any readmission)=%.3f  P(<30d)=%.3f' % (y_any.mean(), y_30.mean()))

# ICD-9 diagnosis codes -> coarse groups
def icd_group(c):
    if pd.isna(c): return 'missing'
    if c[0] in 'VE': return 'other'
    n = float(c)
    if 390 <= n <= 459 or n == 785: return 'circulatory'
    if 460 <= n <= 519 or n == 786: return 'respiratory'
    if 520 <= n <= 579 or n == 787: return 'digestive'
    if int(n) == 250: return 'diabetes'
    if 800 <= n <= 999: return 'injury'
    if 710 <= n <= 739: return 'musculoskeletal'
    if 580 <= n <= 629 or n == 788: return 'genitourinary'
    if 140 <= n <= 239: return 'neoplasms'
    return 'other'
for c in ['diag_1', 'diag_2', 'diag_3']:
    d[c] = d[c].map(icd_group)

d['age_mid'] = d.age.str.extract(r'\[(\d+)-')[0].astype(int) + 5
d['prior_visits'] = d.number_outpatient + d.number_emergency + d.number_inpatient

# ID columns that are really categories
for c in ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']:
    d[c] = d[c].astype(str)
d['payer_code'] = d.payer_code.astype(str)

drug_cols = [c for c in df.columns[24:47] if c not in ('change', 'diabetesMed')]
d['n_drugs_changed'] = d[drug_cols].isin(['Up', 'Down']).sum(axis=1)
d['n_drugs_taken'] = (d[drug_cols] != 'No').sum(axis=1)
drop = ['encounter_id', 'patient_nbr', 'weight', 'readmitted', 'age'] + [c for c in drug_cols if d[c].nunique() < 2]
X = d.drop(columns=drop)
cat = X.select_dtypes(exclude='number').columns.tolist()
num = X.select_dtypes('number').columns.tolist()
X[cat] = X[cat].fillna('missing')
print(len(num), 'numeric,', len(cat), 'categorical features')

## 3. Models (5-fold CV, grouped by patient)

In [ ]:
prep_lin = ColumnTransformer([('n', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), num),
                              ('c', OneHotEncoder(handle_unknown='ignore', min_frequency=20), cat)])
prep_gb = ColumnTransformer([('n', 'passthrough', num),
                             ('c', OneHotEncoder(handle_unknown='ignore', min_frequency=20, sparse_output=False), cat)])
models = {
    'Logistic regression': make_pipeline(prep_lin, LogisticRegression(C=0.05, max_iter=2000)),
    'Gradient boosting': make_pipeline(prep_gb, HistGradientBoostingClassifier(
        max_depth=4, learning_rate=0.05, max_iter=250, l2_regularization=1.0, random_state=RS)),
}
cv = GroupKFold(n_splits=5)

def evaluate(y, name):
    rows, preds = [dict(target=name, model='Baseline (prevalence)', ROC_AUC=0.5, PR_AUC=y.mean())], {}
    for m, pipe in models.items():
        p = cross_val_predict(pipe, X, y, cv=cv, groups=groups, method='predict_proba')[:, 1]
        preds[m] = p
        rows.append(dict(target=name, model=m, ROC_AUC=roc_auc_score(y, p), PR_AUC=average_precision_score(y, p)))
    return pd.DataFrame(rows), preds

r_any, p_any = evaluate(y_any, 'any readmission')
r_30, p_30 = evaluate(y_30, 'readmitted <30d')
pd.concat([r_any, r_30]).round(3).reset_index(drop=True)

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(11, 9))
for row, (y, p, t) in enumerate([(y_any, p_any, 'Any readmission'), (y_30, p_30, 'Readmitted <30 days')]):
    for m, pp in p.items():
        f, tp, _ = roc_curve(y, pp)
        ax[row, 0].plot(f, tp, label=f'{m} ({roc_auc_score(y, pp):.3f})')
        pr, rc, _ = precision_recall_curve(y, pp)
        ax[row, 1].plot(rc, pr, label=m)
    ax[row, 0].plot([0, 1], [0, 1], 'k:')
    ax[row, 0].set(title=t + ' - ROC', xlabel='FPR', ylabel='TPR'); ax[row, 0].legend()
    ax[row, 1].axhline(y.mean(), color='k', ls=':', label='prevalence')
    ax[row, 1].set(title=t + ' - Precision/Recall', xlabel='Recall', ylabel='Precision'); ax[row, 1].legend()
plt.tight_layout(); plt.show()

## 4. What does the model rely on? (permutation importance, held-out patients)

In [ ]:
tr, te = next(GroupShuffleSplit(test_size=0.25, random_state=RS).split(X, y_any, groups))
gb = models['Gradient boosting'].fit(X.iloc[tr], y_any.iloc[tr])
print('hold-out AUC: %.3f' % roc_auc_score(y_any.iloc[te], gb.predict_proba(X.iloc[te])[:, 1]))
pi = permutation_importance(gb, X.iloc[te], y_any.iloc[te], scoring='roc_auc', n_repeats=5, random_state=RS)
s = pd.Series(pi.importances_mean, index=X.columns)
sd = pd.Series(pi.importances_std, index=X.columns)
top = s.sort_values()[-12:].index
s[top].plot.barh(xerr=sd[top], figsize=(7, 5), title='Permutation importance (drop in AUC)'); plt.show()

## 5. Risk stratification: observed rate by predicted-risk decile

In [ ]:
for name, y, p in [('any readmission', y_any, p_any['Gradient boosting']), ('<30d', y_30, p_30['Gradient boosting'])]:
    dec = pd.qcut(p, 10, labels=False)
    print(name)
    print(pd.DataFrame({'mean_pred': p, 'observed': y.values}).groupby(dec).mean().round(3).rename_axis('risk decile').T.to_string(), '\n')

## 6. Conclusions
Results from the run (patient-grouped 5-fold CV):

| Target | Prevalence | Logistic reg. AUC / PR-AUC | Gradient boosting AUC / PR-AUC |
|---|---|---|---|
| Any readmission | 47.1% | 0.680 / 0.651 | **0.690 / 0.661** |
| Readmitted <30 days | 11.4% | 0.663 / 0.215 | **0.672 / 0.229** |

- **Yes, but only weakly.** Both targets are predictable better than chance (AUC 0.67-0.69), but far from reliable at the individual level.
- The models are **well calibrated** and rank patients usefully: the top risk decile has ~78% any-readmission rate vs ~20% in the bottom decile; for <30d it is ~27.5% vs ~4%. That is useful for *prioritising* follow-up, not for confidently flagging individuals.
- A nonlinear model gains only ~0.01 AUC over logistic regression, so the limit is information in the data rather than model complexity.
- Scores are lower than those reported when encounters of the same patient leak across train/test; the grouped CV here is the honest estimate.
- Caveats: no tuning beyond sensible defaults; the label is ordered (`NO`/`>30`/`<30`) and was binarised; data has no information on post-discharge care, which likely drives much of the remaining variance.